# Concordance: the c-statistic, Somers' D, Gini and Kendall's tau

Wiki reference for [Concordance and rank statistics](https://ml-viz-ruby.vercel.app/wiki/concordance-and-rank-statistics).

**The idea in one sentence.** Take one positive and one negative, and ask whether the model scored the positive higher; count that over every such pair and you have the c-statistic, which is the AUC, Somers' D, the Gini coefficient and Kendall's tau all at once, differing only in what they do with **ties**.

**Why it matters.** Every ranking model gets summarised by one of these numbers: a credit scorecard's Gini, a survival model's c-index, `roc_auc_score` in a notebook. Knowing they are one pair count with four tie conventions tells you when they agree (a continuous score) and when they diverge badly (risk grades, rounded scores, shallow trees).

*Colab tip: File → Save a copy in Drive before editing.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score

plt.style.use('dark_background')
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

## 1 — From scratch: counting pairs

Eight loan applicants, scored by a model and then observed. Three defaulted (the positives), five repaid. Note the deliberate tie: applicants C and D both scored 0.65, and one defaulted while the other did not.

In [ ]:
scores = np.array([0.82, 0.71, 0.65, 0.65, 0.50, 0.44, 0.30, 0.12])
labels = np.array([   1,    0,    1,    0,    1,    0,    0,    0])
names  = np.array(list('ABCDEFGH'))

for n, s, y in zip(names, scores, labels):
    print(f'{n}  score {s:.2f}  {"default" if y else "repaid "}')
print(f'\nn_pos = {labels.sum()}, n_neg = {(1 - labels).sum()}, '
      f'pairs = {labels.sum() * (1 - labels).sum()}')

Every (positive, negative) pair is exactly one of three things. The double loop below is the definition, written out literally — `O(n_pos * n_neg)`, which is fine for eight rows and hopeless for a real test set (§4 fixes that).

In [ ]:
def pair_counts(scores, labels):
    """Concordant, discordant and tied (positive, negative) pairs."""
    pos = scores[labels == 1]      # shape (n_pos,)
    neg = scores[labels == 0]      # shape (n_neg,)
    diff = pos[:, None] - neg[None, :]   # shape (n_pos, n_neg)
    C = int((diff > 0).sum())
    D = int((diff < 0).sum())
    T = int((diff == 0).sum())
    return C, D, T

C, D, T = pair_counts(scores, labels)
print(f'C = {C}   D = {D}   T = {T}   total = {C + D + T}')

### The four statistics

Same three counts, four different divisions. The only thing they disagree about is the tied pair:

$$c = \frac{C + T/2}{C + D + T}, \quad D_{xy} = \frac{C - D}{C + D + T}, \quad \tau_a = \frac{C - D}{n(n-1)/2}, \quad \gamma = \frac{C - D}{C + D}$$

In [ ]:
def rank_stats(scores, labels):
    C, D, T = pair_counts(scores, labels)
    n = len(labels)
    return {
        'c':        (C + 0.5 * T) / (C + D + T),   # ties are half a win
        'somers_d': (C - D) / (C + D + T),         # ties count for nothing
        'tau_a':    (C - D) / (n * (n - 1) / 2),   # over ALL pairs of rows
        'gamma':    (C - D) / (C + D),             # ties deleted
    }

st = rank_stats(scores, labels)
for k, v in st.items():
    print(f'{k:>9}: {v:.4f}')

print(f'\nSomers D == 2c - 1 ?  {st["somers_d"]:.4f} vs {2 * st["c"] - 1:.4f}')
print(f'Gini coefficient   =  {2 * st["c"] - 1:.4f}')

## 2 — The library way, cross-checked

`sklearn.metrics.roc_auc_score` sweeps a threshold and integrates; we counted pairs. They must agree to floating point, because the Wilcoxon–Mann–Whitney statistic *is* the area under the ROC curve. `scipy.stats.kendalltau` with `variant='b'` also uses concordance, with its own tie correction.

In [ ]:
auc_sklearn = roc_auc_score(labels, scores)
print(f'roc_auc_score : {auc_sklearn:.10f}')
print(f'our c-statistic: {st["c"]:.10f}')
assert np.allclose(auc_sklearn, st['c']), 'pair count and ROC integration must agree'

tau_b = stats.kendalltau(scores, labels, variant='b')
print(f'\nkendalltau tau-b : {tau_b.statistic:.4f}  (its own tie correction)')
print(f'our tau-a        : {st["tau_a"]:.4f}  (no tie correction, all pairs)')

### The O(n log n) route

Counting pairs is quadratic. The rank-sum identity gets the same number after a single sort, using **midranks** so that tied observations share the average of the ranks they span:

$$c = \frac{R_+ - n_+(n_+ + 1)/2}{n_+ n_-}, \qquad R_+ = \sum_{i:\, y_i = 1} r_i$$

In [ ]:
def c_statistic_fast(scores, labels):
    """Same number, O(n log n): midranks + the rank-sum identity."""
    r = stats.rankdata(scores)          # midranks handle ties automatically
    n_pos = labels.sum()
    n_neg = len(labels) - n_pos
    R_pos = r[labels == 1].sum()
    return (R_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

print('midranks:', stats.rankdata(scores))
print(f'R+ = {stats.rankdata(scores)[labels == 1].sum():.1f}')
print(f'c (rank sum) = {c_statistic_fast(scores, labels):.10f}')
assert np.allclose(c_statistic_fast(scores, labels), st['c'])

big_y = rng.integers(0, 2, 200_000)
big_s = rng.normal(size=200_000) + big_y * 1.2
print(f'\n200k rows, rank-sum c = {c_statistic_fast(big_s, big_y):.4f} '
      f'(the double loop here would be 10^10 comparisons)')

## 3 — Visualize it: what ties do

Now the payoff. Collapse the same eight scores onto a three-grade scorecard (High / Mid / Low), exactly what a risk grade, a rounded score or a shallow tree does. The ordering *between* grades survives; the ordering *within* a grade is destroyed, so within-grade pairs become ties.

In [ ]:
grades = np.where(scores >= 0.65, 3, np.where(scores >= 0.44, 2, 1))
print('grade (3=High, 2=Mid, 1=Low):', grades)

fine   = rank_stats(scores, labels)
coarse = rank_stats(grades.astype(float), labels)

print(f'\n{"":>10} {"raw scores":>12} {"3 grades":>10}')
print(f'{"C/D/T":>10} {str(pair_counts(scores, labels)):>12} '
      f'{str(pair_counts(grades.astype(float), labels)):>10}')
for k in ('c', 'somers_d', 'gamma'):
    print(f'{k:>10} {fine[k]:>12.4f} {coarse[k]:>10.4f}')

In [ ]:
keys = ['c', 'somers_d', 'gamma']
x = np.arange(len(keys))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - 0.2, [fine[k] for k in keys], 0.4, label='raw scores', color='#14b8a6')
ax.bar(x + 0.2, [coarse[k] for k in keys], 0.4, label='three grades', color='#f43f5e')
for i, k in enumerate(keys):
    ax.annotate('', xy=(i + 0.2, coarse[k]), xytext=(i - 0.2, fine[k]),
                arrowprops=dict(arrowstyle='->', color='#eab308', lw=1.5))
ax.set_xticks(x); ax.set_xticklabels(['c-statistic', "Somers' D", 'gamma'])
ax.set_ylim(0, 0.9); ax.set_ylabel('value')
ax.set_title('Binning the score: two statistics fall, one rises')
ax.legend(); plt.tight_layout(); plt.show()

**What to notice.** The c-statistic falls 0.767 → 0.700 and Somers' D falls 0.533 → 0.400: both charge the model for the discrimination it just threw away. Gamma *rises*, 0.571 → 0.600, because it deletes tied pairs from the denominator rather than counting them against the model. A model that refuses to express a preference is rewarded by gamma and penalised by the other two, which is exactly why validation packs quote the c-statistic and not gamma.

## 4 — Tradeoffs and when to use what

| Statistic | Use it when | Avoid it when | Cost |
|---|---|---|---|
| **c-statistic / AUC** | binary outcome, any score | you need calibration too (it is blind to it) | $O(n\log n)$ |
| **Somers' D / Gini** | reporting to a risk audience | comparing across papers that quote AUC | same, it is $2c-1$ |
| **Kendall's tau-a** | two continuous rankings | binary labels (it cannot reach 1) | $O(n\log n)$ |
| **Goodman–Kruskal γ** | almost never on model scores | ties are present, which is when it misleads | $O(n\log n)$ |
| **Harrell's c-index** | survival data with censoring | heavy censoring without IPCW weighting | $O(n\log n)$ |

**Failure modes to remember**

1. **Concordance is invariant to monotone transforms.** A model that ranks perfectly but predicts probabilities 100× too large still scores $c = 1$. Audit calibration separately.
2. **Ties are not rare.** Trees, risk grades and rounded scores all produce them in bulk; the tie convention then decides your headline number.
3. **Small positive counts.** With 40 positives the standard error of $c$ is near 0.05, so a 0.02 gap between two models is noise until a paired test says otherwise.
4. **Class prevalence does not affect $c$**, which is a feature for ranking and a trap for deployment: a flattering AUC at 1% prevalence can still mean an unusable precision.

In [ ]:
# How noisy is c on a small test set? Bootstrap it.
n_pos, n_neg = 40, 960
true_scores = np.concatenate([rng.normal(1.0, 1, n_pos), rng.normal(0.0, 1, n_neg)])
true_labels = np.concatenate([np.ones(n_pos, int), np.zeros(n_neg, int)])

boot = np.array([
    (lambda idx: c_statistic_fast(true_scores[idx], true_labels[idx]))(
        np.concatenate([rng.choice(np.flatnonzero(true_labels == 1), n_pos),
                        rng.choice(np.flatnonzero(true_labels == 0), n_neg)]))
    for _ in range(2000)
])
print(f'point estimate : {c_statistic_fast(true_scores, true_labels):.3f}')
print(f'bootstrap se   : {boot.std(ddof=1):.3f}')
print(f'95% percentile : [{np.percentile(boot, 2.5):.3f}, {np.percentile(boot, 97.5):.3f}]')

## 5 — ✏️ Your turn

### Exercise 1: Somers' D from a pair count

A model produces $C = 180$, $D = 60$, $T = 10$ on its test set. Compute the c-statistic, Somers' D and gamma without using any of the functions above.

In [ ]:
def exercise_1(C, D, T):
    total = None    # TODO(you): how many pairs in total?
    c = None        # TODO(you): ties count as half a win
    somers_d = None # TODO(you): ties count as nothing, but stay in the denominator
    gamma = None    # TODO(you): ties deleted entirely
    return c, somers_d, gamma

# c, somers_d, gamma = exercise_1(180, 60, 10)

In [ ]:
c_, d_, g_ = exercise_1(180, 60, 10)
assert abs(c_ - 0.74) < 1e-9, f'c should be 0.74, got {c_}'
assert abs(d_ - 0.48) < 1e-9, f"Somers' D should be 0.48, got {d_}"
assert abs(g_ - 0.50) < 1e-9, f'gamma should be 0.50, got {g_}'
assert abs(d_ - (2 * c_ - 1)) < 1e-9, "Somers' D must equal 2c - 1"
print('correct')

<details>
<summary>Solution</summary>

```python
def exercise_1(C, D, T):
    total = C + D + T          # 250
    c = (C + 0.5 * T) / total  # 185 / 250 = 0.74
    somers_d = (C - D) / total # 120 / 250 = 0.48 = 2c - 1
    gamma = (C - D) / (C + D)  # 120 / 240 = 0.50
    return c, somers_d, gamma
```

Note that gamma (0.50) exceeds Somers' D (0.48) the moment any ties exist.
</details>

### Exercise 2: the c-index with censoring

Harrell's c-index counts only **comparable** pairs. A pair $(i, j)$ is comparable if you can tell who failed first: subject $i$ had an observed event at $t_i$, and $t_j > t_i$ (whether or not $j$ was later censored). A pair where the earlier of the two times is a censoring time is not comparable.

Fill in the comparability rule, then count concordance among comparable pairs: lower predicted risk should go with longer survival.

In [ ]:
def c_index(risk, time, event):
    """Harrell's c-index. risk: higher = more hazardous. event: 1 = observed."""
    num = den = 0.0
    n = len(time)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            comparable = None   # TODO(you): i must have an observed event, and t_i < t_j
            if not comparable:
                continue
            den += 1
            if risk[i] > risk[j]:
                num += 1
            elif risk[i] == risk[j]:
                num += 0.5
    return num / den

# in the data below, subject 3 is censored at t = 2, before any failure

In [ ]:
den_note = ('5 of the 20 ordered pairs are comparable: subject 3 is censored at '
            't = 2 before anyone failed, so nothing can be said about it, and '
            'subject 4 fails last so it is never the earlier member of a pair.')
time  = np.array([5.0, 6.0, 7.0, 2.0, 9.0])
event = np.array([  1,    1,    0,    0,    1])
risk  = np.array([0.9, 0.7, 0.4, 0.8, 0.1])

got = c_index(risk, time, event)
assert abs(got - 1.0) < 1e-9, f'expected a perfect 1.0 on this data, got {got}'

event2 = event.copy(); risk2 = risk.copy()
risk2[0], risk2[1] = risk2[1], risk2[0]        # swap the two riskiest: one error
got2 = c_index(risk2, time, event2)
assert abs(got2 - 0.8) < 1e-9, f'expected 4/5 after one swap, got {got2}'
print(den_note)
print(f'perfect ranking: {got:.4f}   after one swap: {got2:.4f} = 4/5')

<details>
<summary>Solution</summary>

```python
comparable = event[i] == 1 and time[i] < time[j]
```

Subject 3 (index 3, time 2.0, censored) is never the earlier member of a comparable pair, because we never learn whether it would have failed before the others. That is the whole difference between the c-index and the plain c-statistic: censoring shrinks the denominator, so a c-index on heavily censored data rests on far fewer pairs than the sample size suggests.
</details>

## 6 — Key takeaways

- **One pair count, four statistics.** $C$, $D$ and $T$ are the raw material; the c-statistic, Somers' D, tau-a and gamma differ only in how they treat $T$.
- **$c$ is the AUC**, and **Gini = Somers' D = $2c - 1$**. Three names for one number.
- **Ties are where they diverge.** Binning a score raises $T$; $c$ and Somers' D fall, gamma rises. Compute on raw scores.
- **Use the rank-sum identity** with midranks for anything above a few thousand rows.
- **Concordance says nothing about calibration**, and nothing about whether a gap between two models is real.

**Next:** [Decile analysis, gains and lift](https://ml-viz-ruby.vercel.app/wiki/decile-analysis-and-lift) for where the risk actually landed, and [comparing model performance](https://ml-viz-ruby.vercel.app/wiki/comparing-model-performance) for the paired tests.